<a href="https://colab.research.google.com/github/Ronglawan/PROJECT_PHY_483/blob/main/MINI_PROJECT_PHY_483.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier

from tensorflow import keras
from tensorflow.keras import layers

#Load dataset

In [ ]:
url = "https://raw.githubusercontent.com/Ronglawan/PROJECT_PHY_483/refs/heads/main/ev_battery_qc_data_2026_kaggle.csv"

df = pd.read_csv(url)

df.head()

display(df.head())

In [ ]:
display(pd.DataFrame(df.columns, columns=['Column Name']))

##Data preprocessing

In [ ]:
# ลบคอลัมน์ที่ไม่จำเป็น + กัน data leakage
df = df.drop(['Cell_ID','Batch_ID','Inspector_Comment','Defect_Type'], axis=1)

# จัดการ missing
df = df.dropna()

# Encode
le = LabelEncoder()
df['Production_Line'] = le.fit_transform(df['Production_Line'])
df['Shift'] = le.fit_transform(df['Shift'])
df['Supplier'] = le.fit_transform(df['Supplier'])
df['QC_Grade'] = le.fit_transform(df['QC_Grade'])

##Split data

In [ ]:
X = df.drop('QC_Grade', axis=1)
y = df['QC_Grade']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

##Normalize data

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#Method
##Model 1: Deep Learning

In [ ]:
model = keras.Sequential([
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2
)

## Deep Learning Training History (Accuracy & Loss)

In [ ]:
# Model Accuracy

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['Train','Validation'])
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.show()

#Model Loss

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.legend(['Train','Validation'])
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

##Evaluate DL

In [ ]:
# 1. ทำนายผล (Predict)
y_pred_dl = model.predict(X_test)
y_pred_dl = np.argmax(y_pred_dl, axis=1)

# 2. แสดงค่า Accuracy
print(f"Deep Learning Accuracy: {accuracy_score(y_test, y_pred_dl):.4f}")

# 3. แสดง Classification Report (รายละเอียดความแม่นยำในแต่ละเกรด)
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred_dl))

##Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_dl)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Deep Learning Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()

##Model 2: Random Forest

##Train Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\n=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))

##Confusion Matrix

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(8,6))
sns.heatmap(cm_rf, annot=True, fmt='d')
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Feature Importance

In [ ]:
import pandas as pd

# ดึงค่าความสำคัญ
importance = rf.feature_importances_

# ชื่อ feature
features = X.columns

# สร้างตาราง
imp_df = pd.DataFrame({
    'Feature': features,
    'Importance': importance
})

# เรียงจากมากไปน้อย
imp_df = imp_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)
imp_df.index += 1

display(imp_df)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.barh(imp_df['Feature'], imp_df['Importance'])
plt.gca().invert_yaxis()
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance")
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()

## Model Prediction Preview

In [ ]:
labels = ['Grade A','Grade B','Scrap']
pred_labels = [labels[i] for i in y_pred_rf]

for i in range(10):
    print("Actual:", y_test.iloc[i], "| Predicted:", pred_labels[i])

## Model Performance Comparison

In [ ]:
# สร้างตารางสำหรับ Deep Learning
report_dl = classification_report(y_test, y_pred_dl, output_dict=True)
df_dl = pd.DataFrame(report_dl).transpose()

# สร้างตารางสำหรับ Random Forest
report_rf = classification_report(y_test, y_pred_rf, output_dict=True)
df_rf = pd.DataFrame(report_rf).transpose()

# สร้างตารางสรุปผลเปรียบเทียบ
results = {
    'Metric': ['Accuracy', 'F1-Score (Weighted)'],
    'Random Forest': [
        accuracy_score(y_test, y_pred_rf),
        # ใช้ค่า weighted avg จาก classification report
        classification_report(y_test, y_pred_rf, output_dict=True)['weighted avg']['f1-score']
    ],
    'Deep Learning': [
        accuracy_score(y_test, y_pred_dl),
        # ใช้ค่า weighted avg จาก classification report
        classification_report(y_test, y_pred_dl, output_dict=True)['weighted avg']['f1-score']
    ]
}

df_summary = pd.DataFrame(results)

print("=== Deep Learning Classification Report ===")
display(df_dl)

print("\n=== Random Forest Classification Report ===")
display(df_rf)

print("=== สรุปผลเปรียบเทียบประสิทธิภาพโมเดล ===")
display(df_summary)